<a href="https://colab.research.google.com/github/reigniell/port/blob/main/Agritech_Autonomous_Quality_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# Install Pydantic for advanced data validation
!pip install pydantic

import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field, field_validator
from google.colab import files

In [32]:
print("Please select and upload your downloaded 'flower.csv' file:")
uploaded = files.upload()

# Get the exact filename of your uploaded file
csv_filename = list(uploaded.keys())[0]
print(f"\nSuccessfully uploaded: {csv_filename}")

Please select and upload your downloaded 'flower.csv' file:


Saving flower_dataset.csv to flower_dataset (4).csv

Successfully uploaded: flower_dataset (4).csv


In [65]:
import pandas as pd
import logging
from pydantic import BaseModel, Field
from typing import Literal

# Setup
logging.basicConfig(filename='agritech_audit.log', level=logging.INFO, force=True)

class FlowerValidationSchema(BaseModel):
    species: Literal["Rose", "Shoeblack Plant", "Hibiscus"]
    size: Literal["Small", "Medium", "Large"]
    fragrance: Literal["Low", "Medium", "High"]
    height: float = Field(..., alias="height")

class AutonomousAgronomist:
    @staticmethod
    def generate_remediation_commands(val, score, grade) -> str:
        if "Grade A" in grade: return "NO REMEDIATION: Optimal health."
        commands = []
        if val.fragrance == "Low": commands.append("ACTUATOR_09: Boost UV-B.")
        if val.height < 15.0: commands.append("ACTUATOR_12: Increase nitrogen.")
        return " | ".join(commands) if commands else "SYSTEM_OK"

def run_pipeline(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.lower().str.strip()
    valid_records = []

    for idx, row in df.iterrows():
        raw = row.to_dict()
        raw['species'] = str(raw.get('species', '')).strip().title()
        raw['size'] = str(raw.get('size', '')).strip().title()
        if 'height_cm' in raw: raw['height'] = raw.pop('height_cm')
        raw['fragrance'] = {"None": "Low", "Mild": "Medium", "Strong": "High", "Low": "Low", "Medium": "Medium", "High": "High"}.get(str(raw.get('fragrance', '')).strip().title(), "Low")

        try:
            val = FlowerValidationSchema(**raw)
            score = {"Small": 10, "Medium": 20, "Large": 30}.get(val.size, 0) + \
                    {"Low": 5, "Medium": 15, "High": 35}.get(val.fragrance, 0) + \
                    min(val.height * 0.8, 35)

            grade, price = ("Grade A (Premium Export)", 18.50) if score >= 75 else \
                           ("Grade B (Standard Market)", 9.00) if score >= 45 else \
                           ("Grade C (Processing/Compost)", 2.50)

            # Generate remediation commands
            actions = AutonomousAgronomist.generate_remediation_commands(val, score, grade)

            valid_records.append({
                "species": val.species, "size": val.size, "fragrance": val.fragrance,
                "height": val.height, "score": round(score, 2),
                "grade": grade, "value": price, "actions": actions
            })
        except Exception:
            continue

    return pd.DataFrame(valid_records)

# RUN PIPELINE
processed_df = run_pipeline(csv_filename)

def generate_summary_report(df):
    total_val = df['value'].sum()
    remediation_needed = df[df['actions'] != "NO REMEDIATION: Optimal health."].shape[0]
    report = f"""
    --- AGRITECH ENGINE RUN SUMMARY ---
    Total Crops Analyzed: {len(df)}
    Total Estimated Value: ${total_val:,.2f}
    Crops Requiring Attention: {remediation_needed}
    Status: {'OPERATIONAL - ACTION REQUIRED' if remediation_needed > 0 else 'OPTIMAL'}
    ------------------------------------
    """
    print(report)
    display(df.head(10))

if not processed_df.empty:
    generate_summary_report(processed_df)
else:
    print("CRITICAL: No records passed validation. Check your CSV column names!")


    --- AGRITECH ENGINE RUN SUMMARY ---
    Total Crops Analyzed: 10000
    Total Estimated Value: $112,667.00
    Crops Requiring Attention: 7614
    Status: OPERATIONAL - ACTION REQUIRED
    ------------------------------------
    


,species,size,fragrance,height,score,grade,value,actions
0,Rose,Medium,Medium,48.55,70.0,Grade B (Standard Market),9.0,SYSTEM_OK
1,Shoeblack Plant,Medium,Medium,147.07,70.0,Grade B (Standard Market),9.0,SYSTEM_OK
2,Shoeblack Plant,Medium,Low,102.93,60.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
3,Hibiscus,Large,Low,184.00,70.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
4,Shoeblack Plant,Large,Medium,83.07,80.0,Grade A (Premium Export),18.5,NO REMEDIATION: Optimal health.
5,Hibiscus,Large,Low,120.88,70.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
6,Rose,Small,Medium,99.24,60.0,Grade B (Standard Market),9.0,SYSTEM_OK
7,Rose,Small,High,96.75,80.0,Grade A (Premium Export),18.5,NO REMEDIATION: Optimal health.
8,Hibiscus,Large,Low,160.88,70.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
9,Shoeblack Plant,Large,Medium,60.43,80.0,Grade A (Premium Export),18.5,NO REMEDIATION: Optimal health.


In [66]:
# Download the final, value-added crop data to your local machine
files.download("clean_crop_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [70]:
# View the first 10 lines of your professional audit log
!head -n 10 agritech_audit.log

In [71]:
# Load the raw data again and peek at the row that failed
df = pd.read_csv(csv_filename)
print("The data that caused Row 7 to fail:")
print(df.iloc[7])

The data that caused Row 7 to fail:
species        rose
size          small
fragrance    strong
height_cm     96.75
Name: 7, dtype: object


In [72]:
# Force-delete the old log file so we can start a fresh one
!rm agritech_audit.log

In [73]:
# 1. Re-run the pipeline to make sure we have the data
# This step creates the 'processed_df' variable
run_pipeline(csv_filename)

# 2. Now run the report generator
# This will work because the step above just finished creating 'processed_df'
generate_summary_report(processed_df)


    --- AGRITECH ENGINE RUN SUMMARY ---
    Total Crops Analyzed: 10000
    Total Estimated Value: $112,667.00
    Crops Requiring Attention: 7614
    Status: OPERATIONAL - ACTION REQUIRED
    ------------------------------------
    


,species,size,fragrance,height,score,grade,value,actions
0,Rose,Medium,Medium,48.55,70.0,Grade B (Standard Market),9.0,SYSTEM_OK
1,Shoeblack Plant,Medium,Medium,147.07,70.0,Grade B (Standard Market),9.0,SYSTEM_OK
2,Shoeblack Plant,Medium,Low,102.93,60.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
3,Hibiscus,Large,Low,184.00,70.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
4,Shoeblack Plant,Large,Medium,83.07,80.0,Grade A (Premium Export),18.5,NO REMEDIATION: Optimal health.
5,Hibiscus,Large,Low,120.88,70.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
6,Rose,Small,Medium,99.24,60.0,Grade B (Standard Market),9.0,SYSTEM_OK
7,Rose,Small,High,96.75,80.0,Grade A (Premium Export),18.5,NO REMEDIATION: Optimal health.
8,Hibiscus,Large,Low,160.88,70.0,Grade B (Standard Market),9.0,ACTUATOR_09: Boost UV-B.
9,Shoeblack Plant,Large,Medium,60.43,80.0,Grade A (Premium Export),18.5,NO REMEDIATION: Optimal health.


In [74]:
from google.colab import files
files.download("summary_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [75]:
import plotly.express as px

# Create an interactive pie chart of your Grades
fig = px.pie(processed_df, names='grade', title='Crop Quality Distribution (ROI Overview)')
fig.show()